# Lab: Support Vector Machines, k-Nearest Neighbours and Dimension Reduction

**Master Smart Data, ENSAI** · Duration: 2 hours

This lab compares two classifiers that sit at opposite ends.

* **SVMs are global.** One convex optimisation problem gives one decision function, carried by a few training points: the *support vectors*. With the **kernel trick**, the same linear method draws non-linear boundaries.
* **kNN is local.** There is no training. Each prediction looks at the $k$ closest training points only.

The last part is an opening towards current research. A small linear regression around each point estimates the **gradient** of the regression function. Gradients point towards the few directions that matter, and knowing these directions cures kNN of the *curse of dimensionality*.

| Part | Topic | Data | Time |
|---|---|---|---|
| 0 | Environment check and reminders | | 5 min |
| 1 | SVM and the kernel trick | 2D datasets, circles | 45 min |
| 2 | kNN for classification and regression | `digits`, a 1D signal | 25 min |
| 3 | Opening: gradients and dimension reduction | simulations, `Auto MPG` | 45 min |

Before starting, you must have installed the environment by following the `README.md` of the repository.

**How to read this notebook**

* ✏️ **Paper question**: a short derivation to do with pen and paper.
* 💻 **Coding task**: complete the function or the line marked `# TODO`. Read the docstring first: it gives the inputs, the outputs and the formula.
* ✅ **Test cell**: run it right after your code. It prints `OK` when your implementation is correct.
* 💬 **Question**: write a short answer (two or three sentences) in the cell below it.
* 🎁 **Bonus**: skip it if you are short on time, and come back to it at the end.

Use `numpy` only in your own functions. The SVM solver of `scikit-learn` is used in the provided cells.

**The thread of the lab.** Three functions travel through the lab, so write them carefully. `sq_dists` (Part 1) drives the Gaussian kernel, then kNN. `ridge_fit` comes back from the regression lab. `local_linear` (Part 2) estimates a function and its gradient, and Part 3 is built on it.

---
## Part 0. Environment check and reminders

Run the cell below first. It checks that this notebook runs inside the virtual environment you created, and that the required packages are installed. If it raises an error, go back to the `README.md` (section *Troubleshooting*).

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python      :", sys.version.split()[0])
print("Interpreter :", sys.executable)

if sys.prefix == sys.base_prefix:
    raise RuntimeError(
        "This notebook is NOT running inside a virtual environment.\n"
        "Activate .venv in your terminal, then restart Jupyter from that terminal "
        "(or select the .venv interpreter in VS Code)."
    )

missing = []
for package in ["numpy", "scipy", "matplotlib", "scikit-learn", "pandas"]:
    try:
        print("%-12s: %s" % (package, version(package)))
    except PackageNotFoundError:
        missing.append(package)
if missing:
    raise RuntimeError("Missing packages: %s. Run: pip install -r requirements.txt" % ", ".join(missing))

print("\nEnvironment OK")

In [ ]:
import time
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.svm import SVC
from sklearn.datasets import make_circles, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

plt.rcParams["figure.dpi"] = 100

### Notation

We keep the notation of the regression lab: $m$ examples $(x_i, y_i)$ with $x_i \in \mathbb{R}^d$, stacked in a design matrix $X \in \mathbb{R}^{m \times d}$. In Part 3, $r$ denotes the number of directions that matter, with $r \ll d$.

### Provided functions

`ridge_fit` is the function you wrote in the regression lab. With $\lambda = 0$ it is plain OLS. The two other functions load and plot the 2D datasets of Part 1.

In [ ]:
def ridge_fit(X, y, lam=0.0):
    """Ridge estimator (X^T X + lam I)^{-1} X^T y, from the regression lab.

    X : array of shape (m, d), design matrix (add a column of ones for an intercept).
    y : array of shape (m,).
    lam : float >= 0. lam = 0 gives OLS.
    Returns theta, array of shape (d,).
    """
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)


def load_txt(path):
    """Load a text file with two features and one label (0 or 1) per line."""
    data = np.loadtxt(path, delimiter=",")
    return data[:, :2], data[:, 2]


def plot_svm(decision, X, y, ax, support=None, title="", n_grid=200):
    """Plot a 2D dataset, the decision function of an SVM and its margins.

    decision : function mapping an array (q, 2) of points to an array (q,) of scores.
    support  : indices of the support vectors (circled in black), or None.
    The solid line is {decision = 0}. The dashed lines are {decision = -1} and {decision = +1}.
    """
    pad = 0.05 * (X.max(0) - X.min(0))
    u = np.linspace(X[:, 0].min() - pad[0], X[:, 0].max() + pad[0], n_grid)
    v = np.linspace(X[:, 1].min() - pad[1], X[:, 1].max() + pad[1], n_grid)
    U, V = np.meshgrid(u, v)
    Z = decision(np.column_stack([U.ravel(), V.ravel()])).reshape(U.shape)
    ax.contourf(U, V, Z > 0, alpha=0.15, cmap="coolwarm")
    ax.contour(U, V, Z, levels=[-1, 0, 1], colors="k",
               linestyles=["--", "-", "--"], linewidths=[1, 2, 1])
    ax.scatter(*X[y == 1].T, marker="+", c="tab:red", label="y = 1")
    ax.scatter(*X[y != 1].T, marker="o", s=12, c="tab:blue", label="y = 0")
    if support is not None:
        ax.scatter(*X[support].T, s=80, facecolors="none", edgecolors="k", label="support vectors")
    ax.set_title(title)

---
## Part 1. Support Vector Machines and the kernel trick (45 min)

### 1.1 Reminder

The labels are $y_i \in \{-1, +1\}$. A linear classifier predicts the sign of $w^\top x + b$.

**Hard margin.** If the classes are linearly separable, the SVM chooses the hyperplane that is the farthest from both classes:
$$
\min_{w, b} \ \tfrac12 \|w\|^2 \quad \text{subject to} \quad y_i (w^\top x_i + b) \ge 1 \ \text{ for all } i.
$$
The lines $w^\top x + b = \pm 1$ delimit a band of width $2 / \|w\|$, the **margin**. The points on these lines are the **support vectors**.

**Soft margin.** Real data are rarely separable. We allow violations and pay for them:
$$
\min_{w, b} \ \tfrac12 \|w\|^2 + C \sum_{i=1}^m \max\big(0, \ 1 - y_i (w^\top x_i + b)\big).
$$
The second term is the **hinge loss**. The structure is a penalty on $w$ plus a data-fitting loss, as for Ridge in the regression lab. $C$ plays the role of $1/\lambda$: a large $C$ means weak regularisation.

**The dual solution.** The optimal $w$ is a combination of the training points, $w = \sum_i \alpha_i y_i x_i$ with $0 \le \alpha_i \le C$. The decision function is therefore
$$
h(x) = w^\top x + b = \sum_{i=1}^m \alpha_i y_i \, \langle x_i, x \rangle + b .
$$
Only the support vectors have $\alpha_i > 0$. Keep one fact in mind for Section 1.3: **the data enter the SVM only through inner products** $\langle x_i, x_j \rangle$.

### 1.2 💻 Coding task: linear SVM and the role of $C$

Dataset 1 is almost linearly separable, with one outlier of class 1 on the far left, near $(0.1, 4.1)$.

We give scikit-learn our own matrix of inner products, the **Gram matrix** $K_{ij} = \langle x_i, x_j \rangle$, with the option `kernel="precomputed"`. The SVM then never sees the points themselves, only $K$. To predict at new points $Z$, it needs the inner products between $Z$ and the training points.

💻 **Coding task.** Write `linear_kernel` **without loops**.

In [ ]:
def linear_kernel(X1, X2):
    """Matrix of inner products between the rows of X1 and the rows of X2.

    X1 : array of shape (q, d).
    X2 : array of shape (m, d).
    Returns K, array of shape (q, m), with K[i, j] = <X1[i], X2[j]>.
    """
    # TODO (one line)
    raise NotImplementedError

In [ ]:
# ✅ Test cell
rng = np.random.default_rng(0)
A, B = rng.standard_normal((4, 3)), rng.standard_normal((5, 3))
K = linear_kernel(A, B)
assert K.shape == (4, 5)
assert np.isclose(K[2, 3], A[2] @ B[3])
print("linear_kernel: OK")

In [ ]:
X1, y1 = load_txt("data/data1.txt")
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*X1[y1 == 1].T, marker="+", c="tab:red", label="y = 1")
ax.scatter(*X1[y1 == 0].T, marker="o", s=12, c="tab:blue", label="y = 0")
ax.legend(); ax.set_title("Dataset 1")
plt.show()

💻 **Coding task.** We train the SVM for $C = 1$ and $C = 100$. For each model, recover $w = \sum_i \alpha_i y_i x_i$ from the dual solution, then the margin width $2/\|w\|$. In scikit-learn, `svc.dual_coef_` stores the products $\alpha_i y_i$ (shape `(1, number of support vectors)`) and `svc.support_` stores the indices of the support vectors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, C in zip(axes, [1, 100]):
    svc = SVC(C=C, kernel="precomputed").fit(linear_kernel(X1, X1), y1)
    w = ...        # TODO: w = sum_i alpha_i y_i x_i, over the support vectors
    margin = ...   # TODO: 2 / ||w||
    acc = np.mean(svc.predict(linear_kernel(X1, X1)) == y1)
    plot_svm(lambda Z: svc.decision_function(linear_kernel(Z, X1)), X1, y1, ax,
             support=svc.support_,
             title=f"C = {C}: {len(svc.support_)} SV, margin {margin:.2f}, train acc {acc:.2f}")
axes[0].legend(loc="lower right")
plt.show()

💬 **Question 1.1.** Which value of $C$ classifies the outlier correctly? Which boundary would you trust on new data? Use the analogy with Ridge.

*Your answer:*

💬 **Question 1.2.** Why does the number of support vectors decrease when $C$ increases?

*Your answer:*

### 1.3 The kernel trick, step by step

**The problem.** Below, two classes form concentric circles. No straight line can separate them.

In [ ]:
Xc2, yc2 = make_circles(n_samples=200, noise=0.08, factor=0.4, random_state=0)

svc_lin = SVC(kernel="linear", C=1).fit(Xc2, yc2)
fig, ax = plt.subplots(figsize=(4.5, 4))
plot_svm(svc_lin.decision_function, Xc2, yc2, ax,
         title=f"Linear SVM on the circles: train acc {svc_lin.score(Xc2, yc2):.2f}")
ax.set_aspect("equal")
plt.show()

#### Step 1. Add features by hand

The two classes differ by their distance to the origin. Let us add this information as a third feature:
$$
\varphi(x) = \big(x_1, \ x_2, \ x_1^2 + x_2^2\big) \in \mathbb{R}^3 .
$$
We then run a **linear** SVM on the points $\varphi(x_i)$. This is the general recipe: map the data to a larger *feature space*, and separate them linearly there.

💻 **Coding task.** Write `phi_circle`.

In [ ]:
def phi_circle(X):
    """Feature map (x1, x2) -> (x1, x2, x1^2 + x2^2).

    X : array of shape (q, 2). Returns an array of shape (q, 3).
    """
    # TODO (one line). Hint: np.column_stack
    raise NotImplementedError

In [ ]:
# ✅ Test cell
assert np.allclose(phi_circle(np.array([[1.0, 2.0], [0.0, -3.0]])), [[1, 2, 5], [0, -3, 9]])
print("phi_circle: OK")

The left plot shows the data in the feature space $\mathbb{R}^3$ with the separating plane. The right plot shows the same classifier seen in the original plane: we compute the decision function at $\varphi(z)$ for each point $z$ of a grid.

In [ ]:
svc_phi = SVC(kernel="linear", C=1).fit(phi_circle(Xc2), yc2)
w, b = svc_phi.coef_[0], svc_phi.intercept_[0]

fig = plt.figure(figsize=(11, 4.5))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
F = phi_circle(Xc2)
ax3.scatter(*F[yc2 == 1].T, c="tab:red", marker="+")
ax3.scatter(*F[yc2 == 0].T, c="tab:blue", s=10)
u = np.linspace(-1.2, 1.2, 10)
U, V = np.meshgrid(u, u)
ax3.plot_surface(U, V, -(w[0] * U + w[1] * V + b) / w[2], alpha=0.3, color="gray")
ax3.set_xlabel("$x_1$"); ax3.set_ylabel("$x_2$"); ax3.set_zlabel("$x_1^2 + x_2^2$")
ax3.set_title("Feature space: a plane separates the classes")

ax2 = fig.add_subplot(1, 2, 2)
plot_svm(lambda Z: svc_phi.decision_function(phi_circle(Z)), Xc2, yc2, ax2,
         title=f"Back in the plane: train acc {svc_phi.score(phi_circle(Xc2), yc2):.2f}")
ax2.set_aspect("equal")
plt.tight_layout()
plt.show()

💬 **Question 1.3.** The SVM is linear in $\mathbb{R}^3$. Why is its boundary a circle in the original plane? Write the equation of the boundary in terms of $x_1, x_2$.

*Your answer:*

#### Step 2. Compute inner products without computing the features

Step 1 works, but we had to guess the right feature. A safer choice is to include **all** monomials up to degree 2:
$$
\varphi_2(x) = \big(1, \ \sqrt2\, x_1, \ \sqrt2\, x_2, \ x_1^2, \ \sqrt2\, x_1 x_2, \ x_2^2\big) \in \mathbb{R}^6 .
$$
Recall from 1.1 that the SVM only needs inner products. So we only need $\langle \varphi_2(x), \varphi_2(x') \rangle$.

✏️ **Paper question 1.4.** Show that for all $x, x' \in \mathbb{R}^2$,
$$
\langle \varphi_2(x), \varphi_2(x') \rangle = \big(1 + x^\top x'\big)^2 .
$$
What is the cost of each side of the equality, in number of operations?

*Your answer:*

The function $K(x, x') = (1 + x^\top x')^2$ is called the **polynomial kernel** of degree 2. Using a kernel instead of explicit features is the **kernel trick**.

💻 **Coding task.** Write `poly_kernel` for any degree $p$, $K(x, x') = (1 + x^\top x')^p$, with `linear_kernel`. The function `phi_poly2` is provided to test it.

In [ ]:
def phi_poly2(X):
    """Explicit degree-2 feature map of Paper question 1.4. X : (q, 2) -> (q, 6)."""
    x1, x2, s = X[:, 0], X[:, 1], np.sqrt(2)
    return np.column_stack([np.ones(len(X)), s * x1, s * x2, x1**2, s * x1 * x2, x2**2])


def poly_kernel(X1, X2, degree=2):
    """Polynomial kernel (1 + <x, x'>)^degree between the rows of X1 and the rows of X2.

    Returns an array of shape (q, m).
    """
    # TODO (one line)
    raise NotImplementedError

In [ ]:
# ✅ Test cell: the kernel equals the inner product of the explicit features
rng = np.random.default_rng(1)
A, B = rng.standard_normal((5, 2)), rng.standard_normal((7, 2))
assert np.allclose(poly_kernel(A, B), phi_poly2(A) @ phi_poly2(B).T)
assert np.isclose(poly_kernel(A, B, degree=3)[0, 0], (1 + A[0] @ B[0]) ** 3)
print("poly_kernel: OK")

**The decisive experiment.** We train two SVMs on the circles:

* **(a)** a *linear* SVM on the explicit features $\varphi_2(x_i) \in \mathbb{R}^6$,
* **(b)** an SVM that only receives the Gram matrix $K_{ij} = (1 + x_i^\top x_j)^2$, computed in $\mathbb{R}^2$.

If the kernel trick is correct, they solve the same optimisation problem and must give the same classifier. Run the cell and compare.

In [ ]:
svc_a = SVC(kernel="linear", C=1).fit(phi_poly2(Xc2), yc2)                 # explicit features
svc_b = SVC(kernel="precomputed", C=1).fit(poly_kernel(Xc2, Xc2), yc2)     # kernel only

Z = np.random.default_rng(2).uniform(-1.3, 1.3, (5000, 2))                # new points
h_a = svc_a.decision_function(phi_poly2(Z))
h_b = svc_b.decision_function(poly_kernel(Z, Xc2))
print("Same support vectors          :", np.array_equal(svc_a.support_, svc_b.support_))
print("Max |h_a - h_b| on 5000 points: %.1e" % np.max(np.abs(h_a - h_b)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
plot_svm(lambda P: svc_a.decision_function(phi_poly2(P)), Xc2, yc2, axes[0],
         support=svc_a.support_, title="(a) linear SVM on explicit features in R^6")
plot_svm(lambda P: svc_b.decision_function(poly_kernel(P, Xc2)), Xc2, yc2, axes[1],
         support=svc_b.support_, title="(b) SVM on the kernel, computed in R^2")
for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

#### Step 3. Why it matters: the cost of explicit features

For the circles, 6 features are cheap. But the number of monomials of degree at most $p$ in $d$ variables is $\binom{d + p}{p}$. The cell below prints it for $d = 2$, for the $8 \times 8$ images of `digits` ($d = 64$) and for MNIST ($d = 784$). It then times the computation of the same Gram matrix on 300 `digits` images, with explicit features and with the kernel.

In [ ]:
print("Number of explicit features, binom(d + p, p):")
print(f"{'':>8}" + "".join(f"{'p = ' + str(p):>14}" for p in [2, 3, 5, 10]))
for d in [2, 64, 784]:
    print(f"{'d = ' + str(d):>8}" + "".join(f"{comb(d + p, p):>14.3g}" for p in [2, 3, 5, 10]))

Xs = load_digits().data[:300] / 16
print("\nGram matrix of 300 digits images:")
for p in [2, 3]:
    t0 = time.perf_counter()
    F = PolynomialFeatures(degree=p).fit_transform(Xs)    # explicit monomials
    G_explicit = F @ F.T
    t1 = time.perf_counter()
    G_kernel = poly_kernel(Xs, Xs, degree=p)              # kernel trick
    t2 = time.perf_counter()
    print(f"  degree {p}: {F.shape[1]:>6} features, explicit {t1 - t0:.4f} s, "
          f"kernel {t2 - t1:.4f} s, ratio {(t1 - t0) / (t2 - t1):.0f}")

💬 **Question 1.5.** Summarise what the two last experiments show. Why can we use a polynomial kernel of degree 10 on MNIST, while the explicit features are out of reach?

*Note.* The explicit features of `PolynomialFeatures` do not have the $\sqrt2$ factors, so `G_explicit` differs slightly from `G_kernel`. The cost, which is what we measure here, is the same.

*Your answer:*

### 1.4 The Gaussian kernel: an infinite feature space

The most used kernel is the **Gaussian kernel**
$$
K(x, x') = \exp\left( - \frac{\|x - x'\|^2}{2 \sigma^2} \right).
$$
It measures similarity: it equals 1 when $x = x'$ and decays to 0 when the points are far apart. The bandwidth $\sigma$ sets what "far" means.

Which features does it correspond to? Expanding $\|x - x'\|^2 = \|x\|^2 + \|x'\|^2 - 2 x^\top x'$ and the exponential gives
$$
K(x, x') = e^{-\|x\|^2 / 2\sigma^2} \, e^{-\|x'\|^2 / 2\sigma^2} \sum_{p=0}^{\infty} \frac{1}{p!} \left( \frac{x^\top x'}{\sigma^2} \right)^p .
$$
It contains polynomial terms **of every degree**: the feature space is infinite-dimensional. Explicit features are impossible here. Only the kernel trick makes this SVM computable.

💻 **Coding task.** First write `sq_dists`, which returns all the squared distances **without loops**, with the identity above:
$$
\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2\, a^\top b .
$$
Rounding errors can make tiny distances slightly negative: clip them at 0 with `np.maximum`. Then write `gaussian_kernel` with `sq_dists`. We will reuse `sq_dists` for kNN in Part 2.

In [ ]:
def sq_dists(X1, X2):
    """Squared Euclidean distances between the rows of X1 and the rows of X2.

    X1 : array of shape (q, d).
    X2 : array of shape (m, d).
    Returns D, array of shape (q, m), with D[i, j] = ||X1[i] - X2[j]||^2.
    """
    # TODO (two lines). Hint: (X1**2).sum(axis=1)[:, None] has shape (q, 1)
    raise NotImplementedError


def gaussian_kernel(X1, X2, sigma=0.1):
    """Gaussian kernel exp(-||x - x'||^2 / (2 sigma^2)). Returns an array of shape (q, m)."""
    # TODO (one line)
    raise NotImplementedError

In [ ]:
# ✅ Test cell
rng = np.random.default_rng(3)
A, B = rng.standard_normal((6, 4)), rng.standard_normal((7, 4))
assert np.allclose(sq_dists(A, B), cdist(A, B, "sqeuclidean"))
assert np.allclose(np.diag(gaussian_kernel(A, A, sigma=0.5)), 1.0)
assert np.isclose(gaussian_kernel(A, B, sigma=2.0)[1, 2], np.exp(-np.sum((A[1] - B[2])**2) / 8))
print("sq_dists and gaussian_kernel: OK")

**What the SVM sees.** Dataset 2 cannot be separated by a line. Below, we sort 300 of its points by class and display the Gram matrix $K_{ij}$ for three bandwidths. Remember that this matrix is **all** the SVM knows about the data. The cell also prints two indicators computed on the whole dataset: the average number of points that each point finds "similar" ($K > 0.5$), and the share of its similarity that comes from points of its own class.

In [ ]:
X2, y2 = load_txt("data/data2.txt")
X2_tr, X2_va, y2_tr, y2_va = train_test_split(X2, y2, test_size=0.3, random_state=0)

idx = np.random.default_rng(4).choice(len(X2), 300, replace=False)
idx = idx[np.argsort(y2[idx], kind="stable")]           # class 0 first, then class 1
n0 = np.sum(y2[idx] == 0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, s in zip(axes, [0.01, 0.05, 1.0]):
    im = ax.imshow(gaussian_kernel(X2[idx], X2[idx], s), vmin=0, vmax=1, cmap="viridis")
    ax.axhline(n0 - 0.5, c="w", lw=1); ax.axvline(n0 - 0.5, c="w", lw=1)
    ax.set_title(f"Gram matrix, σ = {s}"); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

same_class = y2[:, None] == y2[None, :]
print(f"{'sigma':>6} {'similar points (K > 0.5)':>26} {'share from same class':>23}")
for s in [0.01, 0.05, 0.2, 1.0]:
    K = gaussian_kernel(X2, X2, s)
    np.fill_diagonal(K, 0)                                # ignore each point itself
    n_similar = np.mean(np.sum(K > 0.5, axis=1))
    share = np.mean(np.sum(K * same_class, axis=1) / np.sum(K, axis=1))
    print(f"{s:>6} {n_similar:>26.1f} {share:>23.2f}")

💬 **Question 1.6.** Describe the three matrices with the help of the two indicators. The white lines separate the two classes. What does the SVM "know" in each case? Which bandwidth seems the most useful?

*Your answer:*

**Check and reconstruct.** scikit-learn has a built-in Gaussian kernel, `kernel="rbf"`, with $\gamma = 1/(2\sigma^2)$. We check that our kernel gives the same classifier.

💻 **Coding task.** Then rebuild the decision function by hand from the dual solution:
$$
h(x) = \sum_{i \in \text{SV}} \alpha_i y_i \, K(x_i, x) + b ,
$$
using `svc.dual_coef_` (the $\alpha_i y_i$), `svc.support_` (the indices of the support vectors) and `svc.intercept_` (the $b$). Compute $h$ on the validation points.

In [ ]:
sigma, C = 0.1, 1.0
ours = SVC(C=C, kernel="precomputed").fit(gaussian_kernel(X2_tr, X2_tr, sigma), y2_tr)
theirs = SVC(C=C, kernel="rbf", gamma=1 / (2 * sigma**2)).fit(X2_tr, y2_tr)
same = np.mean(ours.predict(gaussian_kernel(X2_va, X2_tr, sigma)) == theirs.predict(X2_va))
print(f"Agreement between our kernel and scikit-learn's 'rbf': {same:.0%}")

X_sv = ...       # TODO: the support vectors
h_manual = ...   # TODO: sum_i alpha_i y_i K(x_i, x) + b, for all validation points at once

In [ ]:
# ✅ Test cell
assert np.allclose(h_manual, ours.decision_function(gaussian_kernel(X2_va, X2_tr, sigma)))
print(f"Decision function rebuilt by hand: OK. It uses {len(X_sv)} support vectors "
      f"out of {len(X2_tr)} training points.")

Now we explore a grid of $(C, \sigma)$ with the built-in kernel, which is faster. Each title gives the training accuracy, the validation accuracy and the number of support vectors.

In [ ]:
Cs, sigmas = [1, 100], [0.01, 0.05, 0.2]
fig, axes = plt.subplots(len(Cs), len(sigmas), figsize=(14, 8.5))
for i, C in enumerate(Cs):
    for j, s in enumerate(sigmas):
        svc = SVC(C=C, kernel="rbf", gamma=1 / (2 * s**2)).fit(X2_tr, y2_tr)
        plot_svm(svc.decision_function, X2_tr, y2_tr, axes[i, j], n_grid=150,
                 title=f"C={C}, σ={s}: train {svc.score(X2_tr, y2_tr):.2f}, "
                       f"val {svc.score(X2_va, y2_va):.2f}, {len(svc.support_)} SV")
plt.tight_layout()
plt.show()

💬 **Question 1.7.** Describe the boundary when $\sigma$ is small, then large, and relate it to Question 1.6. For $\sigma = 0.01$, why is almost every training point a support vector? Which pair $(C, \sigma)$ would you choose, and on which criterion?

*Your answer:*

### 1.5 🎁 Bonus: a real dataset, breast cancer diagnosis

The Wisconsin dataset describes 569 tumours with 30 features computed from images. The label says whether the tumour is malignant or benign. The features have very different scales, from $10^{-2}$ to $10^3$.

💻 **Coding task.** Build a pipeline that standardises the features, then fits a Gaussian SVM. Choose $(C, \gamma)$ by 5-fold cross-validation with `GridSearchCV` on the grid below. Report the test accuracy.

In [ ]:
Xb, yb = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=0, stratify=yb)

raw = SVC(kernel="rbf").fit(Xb_tr, yb_tr)
print(f"Gaussian SVM on raw features    : test accuracy {raw.score(Xb_te, yb_te):.3f}")

param_grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [1e-3, 1e-2, 1e-1, 1]}
pipe = ...     # TODO: StandardScaler followed by SVC(kernel="rbf"), with make_pipeline
search = ...   # TODO: GridSearchCV(pipe, param_grid, cv=5), fitted on the training set
print(f"Standardised and cross-validated: test accuracy {search.score(Xb_te, yb_te):.3f}",
      search.best_params_)

💬 **Question 1.8.** Why does standardisation matter so much for a Gaussian SVM? Would it matter for kNN?

*Your answer:*

---
## Part 2. k-Nearest Neighbours (25 min)

### 2.1 💻 Coding task: kNN classification of handwritten digits

The kNN rule is the simplest classifier one can imagine. To classify $x$:

1. find the $k$ training points closest to $x$, written $N_k(x)$,
2. predict the most frequent label among them.

It estimates $P(Y = c \mid X = x)$ by the proportion of class $c$ in $N_k(x)$. There is no training, only a stored dataset.

We use the `digits` dataset of the regression lab: 1797 images of $8 \times 8$ pixels, so $d = 64$. All pixels share the same scale (0 to 16), so no standardisation is needed.

💻 **Coding task.** Use `sq_dists` from Part 1, then `np.argsort` along `axis=1`. For the vote, `np.bincount(labels).argmax()` returns the most frequent value of an array of non-negative integers.

In [ ]:
def knn_classify(X_train, y_train, X_test, k):
    """kNN classifier with the Euclidean distance and a majority vote.

    X_train : array of shape (m, d).
    y_train : array of shape (m,), non-negative integer labels.
    X_test  : array of shape (q, d).
    k       : number of neighbours.
    Returns y_pred, array of shape (q,).
    """
    # TODO (three lines)
    raise NotImplementedError

In [ ]:
# ✅ Test cell: three points of class 0 near the origin, two points of class 1 far away
Xt = np.array([[0, 0], [0, 1], [1, 0], [10, 10], [10, 11]], dtype=float)
yt = np.array([0, 0, 0, 1, 1])
assert np.array_equal(knn_classify(Xt, yt, np.array([[0.2, 0.2], [9, 9]]), k=1), [0, 1])
assert np.array_equal(knn_classify(Xt, yt, np.array([[9, 9]]), k=5), [0])   # the majority wins
print("knn_classify: OK")

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)

ks = [1, 3, 5, 10, 20, 50, 100]
acc_tr = [np.mean(knn_classify(Xd_tr, yd_tr, Xd_tr, k) == yd_tr) for k in ks]
acc_te = [np.mean(knn_classify(Xd_tr, yd_tr, Xd_te, k) == yd_te) for k in ks]

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(ks, acc_tr, "o-", label="train")
ax.plot(ks, acc_te, "s-", label="test")
ax.set_xscale("log"); ax.set_xlabel("k"); ax.set_ylabel("accuracy"); ax.legend()
ax.set_title("kNN on digits")
plt.show()

svm_digits = SVC(kernel="rbf", gamma="scale").fit(Xd_tr, yd_tr)
print(f"Best kNN test accuracy: {max(acc_te):.3f}   Gaussian SVM: {svm_digits.score(Xd_te, yd_te):.3f}")

💬 **Question 2.1.** Why is the training accuracy exactly 1 for $k = 1$? Which SVM of Part 1 behaves in the same way?

*Your answer:*

💬 **Question 2.2.** Describe the bias-variance trade-off as $k$ varies. Which values of $k$ give a complex model? What happens for $k = m$?

*Your answer:*

### 2.2 💻 Coding task: from kNN to local linear regression

kNN also works for regression: predict the **mean** of the responses of the $k$ neighbours,
$$
\hat f(x) = \frac{1}{k} \sum_{i \in N_k(x)} y_i .
$$
This fits a **constant** locally. We can fit a **line** locally instead. If $f$ is smooth, a first-order Taylor expansion around $x_0$ gives, for $x_i$ close to $x_0$,
$$
y_i \approx f(x_0) + \nabla f(x_0)^\top (x_i - x_0) + \text{noise}.
$$
This is a linear model in the unknowns $f(x_0)$ and $\nabla f(x_0)$. So we run **OLS on the neighbours of $x_0$**, with the design matrix
$$
Z = \begin{pmatrix} 1 & (x_{i_1} - x_0)^\top \\ \vdots & \vdots \\ 1 & (x_{i_k} - x_0)^\top \end{pmatrix} \in \mathbb{R}^{k \times (1 + d)} .
$$
The intercept estimates $f(x_0)$. The slope estimates the **gradient** $\nabla f(x_0)$. Part 3 is built on this second fact.

💻 **Coding task.** Write `knn_regress`, then `local_linear` with `ridge_fit`. The parameter `lam` is only useful in Section 3.4. Keep `lam = 0` (OLS) until then.

In [ ]:
def knn_regress(X_train, y_train, X_test, k):
    """kNN regression: mean response of the k nearest neighbours. Returns an array of shape (q,)."""
    # TODO (two lines)
    raise NotImplementedError


def local_linear(X_train, y_train, x0, k, lam=0.0):
    """Local linear regression at x0 on its k nearest neighbours.

    X_train : array of shape (m, d).
    y_train : array of shape (m,).
    x0      : array of shape (d,), the point of interest.
    k       : number of neighbours (it must exceed d + 1 when lam = 0).
    lam     : Ridge penalty passed to ridge_fit.
    Returns (f_hat, grad_hat): a float estimating f(x0), and an array of shape (d,)
    estimating the gradient of f at x0.
    """
    # TODO (four lines). Hint: x0[None, :] has shape (1, d)
    raise NotImplementedError

In [ ]:
# ✅ Test cell: on an exactly linear function, local linear regression is exact
rng = np.random.default_rng(5)
Xt = rng.standard_normal((50, 3))
a = np.array([1.0, -2.0, 0.5])
yt = 3 + Xt @ a
f0, g0 = local_linear(Xt, yt, Xt[0], k=10)
assert np.isclose(f0, yt[0]) and np.allclose(g0, a)
assert np.isclose(knn_regress(Xt, yt, Xt[:1], k=1)[0], yt[0])
print("knn_regress and local_linear: OK")

We compare both estimators on $f(x) = \sin(2\pi x)$, with $m = 200$ noisy observations on $[0, 1]$ and $k = 25$.

In [ ]:
rng = np.random.default_rng(6)
m, k = 200, 25
x = np.sort(rng.uniform(0, 1, m))
y = np.sin(2 * np.pi * x) + 0.2 * rng.standard_normal(m)
Xr = x[:, None]                                  # shape (m, 1): one feature
grid = np.linspace(0, 1, 200)

f_knn = knn_regress(Xr, y, grid[:, None], k)
fits = [local_linear(Xr, y, np.array([g]), k) for g in grid]
f_ll = np.array([fit[0] for fit in fits])
slope_ll = np.array([fit[1][0] for fit in fits])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x, y, s=8, c="gray", alpha=0.6, label="data")
axes[0].plot(grid, np.sin(2 * np.pi * grid), "k", lw=2, label="true f")
axes[0].plot(grid, f_knn, label="kNN mean")
axes[0].plot(grid, f_ll, label="local linear")
axes[0].legend(); axes[0].set_title(f"Estimating f (k = {k})")
axes[1].plot(grid, 2 * np.pi * np.cos(2 * np.pi * grid), "k", lw=2, label="true f'")
axes[1].plot(grid, slope_ll, c="tab:orange", label="local slope")
axes[1].legend(); axes[1].set_title("Estimating the derivative f'")
plt.show()

💬 **Question 2.3.** Look at the kNN mean near $x = 0$ and $x = 1$. Explain the error. Why does local linear regression reduce it?

*Your answer:*

💬 **Question 2.4.** We obtained an estimate of $f'$ without ever differentiating $f$. Where is the derivative hidden in the procedure?

*Your answer:*

---
## Part 3. Opening: gradients reveal the directions that matter (45 min)

This part introduces a family of methods called **sufficient dimension reduction**. You need no prior knowledge: everything is built from `local_linear`.

### 3.1 The curse of dimensionality

We simulate $x \sim \mathcal N(0, I_d)$ and
$$
y = \sin(2\, \beta^\top x) + \varepsilon, \qquad \|\beta\| = 1 .
$$
The response depends on **one** direction only, whatever $d$. Yet we give kNN the full vector $x \in \mathbb{R}^d$. Run the cell and look at the test error as $d$ grows.

In [ ]:
rng = np.random.default_rng(7)
dims, mse = [1, 2, 5, 10, 20, 50], []
for d in dims:
    beta = rng.standard_normal(d); beta /= np.linalg.norm(beta)
    X = rng.standard_normal((1500, d))
    f_true = np.sin(2 * X @ beta)
    y = f_true + 0.1 * rng.standard_normal(1500)
    X_tr, X_te, y_tr, f_te = X[:1000], X[1000:], y[:1000], f_true[1000:]
    mse.append(np.mean((knn_regress(X_tr, y_tr, X_te, k=10) - f_te) ** 2))

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(dims, mse, "o-", label="kNN, k = 10")
ax.axhline(np.var(f_te), ls="--", c="gray", label="predicting the mean")
ax.set_xlabel("dimension d"); ax.set_ylabel("test MSE"); ax.legend()
plt.show()

💬 **Question 3.1.** The response depends on a single direction. Why does kNN still fail when $d$ grows?

*Your answer:*

### 3.2 Why gradients reveal the relevant directions

We assume a **multi-index model**:
$$
y = f(x) + \varepsilon, \qquad f(x) = g(B^\top x),
$$
where $B \in \mathbb{R}^{d \times r}$ has orthonormal columns and $r \ll d$. The function $g : \mathbb{R}^r \to \mathbb{R}$ is unknown. In words: $f$ depends on $x$ only through $r$ linear combinations of its coordinates. The model of 3.1 is the case $r = 1$, $B = \beta$.

Our goal is to estimate the subspace $\mathrm{span}(B)$ without knowing $g$.

✏️ **Paper question 3.2.**

1. Use the chain rule to show that $\nabla f(x) = B \, \nabla g(B^\top x)$. Deduce that $\nabla f(x) \in \mathrm{span}(B)$ for **every** $x$.
2. Define the $d \times d$ matrix
$$
M = \mathbb{E}\big[ \nabla f(X) \, \nabla f(X)^\top \big].
$$
Show that $M = B\, \mathbb{E}\big[\nabla g \, \nabla g^\top\big] B^\top$. Deduce that $M$ has rank at most $r$, and that $M v = 0$ for every $v$ orthogonal to $\mathrm{span}(B)$.
3. Conclude: how can we recover $\mathrm{span}(B)$ from $M$?
4. Give a geometric interpretation: along which directions is $f$ constant?

*Your answer:*

**Visual check in 2D.** Take $d = 2$, $r = 1$ and $f(x) = \sin(2 \beta^\top x)$ with $\beta$ at 30°. The level sets of $f$ are straight lines orthogonal to $\beta$. We estimate the gradient at 60 sample points with `local_linear`, from the noisy data only. All the arrows should be parallel to $\beta$, up to sign and length.

In [ ]:
rng = np.random.default_rng(8)
angle = np.deg2rad(30)
beta2 = np.array([np.cos(angle), np.sin(angle)])
X = rng.uniform(-2, 2, (600, 2))
y = np.sin(2 * X @ beta2) + 0.05 * rng.standard_normal(600)

pts = X[:60]
grads = np.array([local_linear(X, y, x0, k=30)[1] for x0 in pts])

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(*X.T, c=y, cmap="coolwarm", s=10, alpha=0.6)
ax.quiver(*pts.T, *grads.T, angles="xy", scale_units="xy", scale=4, width=0.005, color="k")
ax.arrow(0, 0, 1.5 * beta2[0], 1.5 * beta2[1], color="tab:green", width=0.04)
ax.set_aspect("equal"); ax.set_title("Estimated gradients (black) and β (green)")
plt.show()

### 3.3 💻 Coding task: the Outer Product of Gradients (OPG) estimator

Paper question 3.2 gives an algorithm. We replace the expectation by an empirical mean and the true gradients by local linear estimates.

1. For each sample $x_i$, estimate $\hat g_i \approx \nabla f(x_i)$ with `local_linear`.
2. Form $\hat M = \frac{1}{m} \sum_{i=1}^m \hat g_i \hat g_i^\top = \frac1m G^\top G$, where $G \in \mathbb{R}^{m \times d}$ stacks the $\hat g_i^\top$ as rows.
3. Eigen-decompose $\hat M$ and keep the $r$ eigenvectors with the largest eigenvalues: they form $\hat B \in \mathbb{R}^{d \times r}$.

This is the **OPG** method (Xia, Tong, Li and Zhu, 2002). The eigenvalues also help to choose $r$: look for a gap.

💻 **Coding task.** Write `opg`. Note that `np.linalg.eigh` returns the eigenvalues of a symmetric matrix in **increasing** order: reorder them.

In [ ]:
def opg(X, y, k, r, lam=0.0):
    """Outer Product of Gradients estimator of the dimension reduction subspace.

    X   : array of shape (m, d).
    y   : array of shape (m,).
    k   : number of neighbours of each local linear regression.
    r   : number of directions to keep.
    lam : Ridge penalty of the local regressions.
    Returns (eigvals, B_hat): the eigenvalues of M_hat in decreasing order, shape (d,),
    and the r leading eigenvectors as columns, shape (d, r).
    """
    # TODO (five lines)
    raise NotImplementedError

To compare two subspaces, we cannot compare bases, which are not unique (signs, rotations). Orthogonal projectors are unique, so we use
$$
\mathrm{dist}(B, \hat B) = \big\| B B^\top - \hat B \hat B^\top \big\|_{\mathrm{op}} \in [0, 1],
$$
which is 0 when the subspaces coincide and 1 when one of them contains a direction orthogonal to the other.

In [ ]:
def subspace_distance(B1, B2):
    # Operator norm of the difference of the two orthogonal projectors
    return np.linalg.norm(B1 @ B1.T - B2 @ B2.T, ord=2)

# ✅ Test cell
# 1. On the 2D example of 3.2, OPG should find beta2
_, b_hat = opg(X, y, k=30, r=1)
print("estimated direction:", b_hat.ravel(), "  true:", beta2)
assert subspace_distance(b_hat, beta2[:, None]) < 0.1
# 2. On an exactly linear function in R^3, all gradients equal a, so OPG must return a / ||a||
rng = np.random.default_rng(9)
X3 = rng.standard_normal((200, 3)); a3 = np.array([1.0, 2.0, 2.0]) / 3
ev3, b3 = opg(X3, X3 @ a3, k=10, r=1)
assert b3.shape == (3, 1) and np.all(np.diff(ev3) <= 1e-12)
assert subspace_distance(b3, a3[:, None]) < 1e-6
print("opg: OK")

Now a harder problem: $d = 10$, $r = 2$, $m = 600$ and
$$
f(x) = \sin\big(1.5\, b_1^\top x\big) + 0.5\, \big(b_2^\top x\big)^2,
\qquad b_1 = \tfrac{1}{\sqrt2}(1, 1, 0, \dots, 0), \quad b_2 = \tfrac{1}{\sqrt3}(0, 0, 1, -1, 1, 0, \dots, 0).
$$
Each local regression has $d + 1 = 11$ unknowns, so we take $k = 50$ neighbours.

In [ ]:
d, m = 10, 600
B_true = np.zeros((d, 2))
B_true[:2, 0] = 1 / np.sqrt(2)
B_true[2:5, 1] = np.array([1, -1, 1]) / np.sqrt(3)

def f_multi(X):
    Z = X @ B_true
    return np.sin(1.5 * Z[:, 0]) + 0.5 * Z[:, 1] ** 2

rng = np.random.default_rng(10)
X_tr = rng.standard_normal((m, d)); y_tr = f_multi(X_tr) + 0.1 * rng.standard_normal(m)
X_te = rng.standard_normal((1000, d)); f_te = f_multi(X_te)

eigvals, B_hat = opg(X_tr, y_tr, k=50, r=2)
print(f"distance between span(B) and span(B_hat): {subspace_distance(B_true, B_hat):.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(np.arange(1, d + 1), eigvals / eigvals.sum())
ax.set_xlabel("index"); ax.set_ylabel("share of the trace"); ax.set_title("Eigenvalues of M_hat")
plt.show()

The payoff: kNN on the 10 raw features, on the 2 estimated features $\hat B^\top x$, and on the 2 true features $B^\top x$. The last one is an oracle, never available in practice.

In [ ]:
print(f"{'k':>4} {'raw (d=10)':>12} {'OPG (r=2)':>12} {'oracle':>10}")
for k in [5, 10, 20]:
    raw = np.mean((knn_regress(X_tr, y_tr, X_te, k) - f_te) ** 2)
    red = np.mean((knn_regress(X_tr @ B_hat, y_tr, X_te @ B_hat, k) - f_te) ** 2)
    ora = np.mean((knn_regress(X_tr @ B_true, y_tr, X_te @ B_true, k) - f_te) ** 2)
    print(f"{k:>4} {raw:>12.3f} {red:>12.3f} {ora:>10.3f}")
print(f"variance of f: {np.var(f_te):.3f}")

💬 **Question 3.3.**

1. How many eigenvalues stand out? Is it consistent with the model?
2. Compare the three columns of the table. Why does the dimension reduction help kNN so much?
3. Why must $k$ exceed $d + 1$ in `local_linear` with `lam = 0`? What does it imply when $d$ is large?

*Your answer:*

### 3.4 A real dataset: fuel consumption of cars

The *Auto MPG* dataset (UCI repository) describes 392 car models sold between 1970 and 1982. The response is the fuel efficiency in miles per gallon (`mpg`). We use six features: `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration` and `model_year`. The file `data/mpg.csv` also contains the origin and the name of each car, which we drop. Six cars have a missing `horsepower` and are removed.

`cylinders` and `model_year` take few distinct values. Inside a small neighbourhood such a column is almost constant, so $Z^\top Z$ is singular or badly conditioned. This is the situation of the regression lab where Ridge saved OLS. We set `lam = 1`.

As always, the standardisation and OPG are fitted on the training set only.

In [ ]:
cars = pd.read_csv("data/mpg.csv").dropna()
features = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
Xc, yc = cars[features].to_numpy(dtype=float), cars["mpg"].to_numpy(dtype=float)
print(f"{len(yc)} cars, {len(features)} features")

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0)
mu, sd = Xc_tr.mean(0), Xc_tr.std(0)
Zc_tr, Zc_te = (Xc_tr - mu) / sd, (Xc_te - mu) / sd    # standardise with training statistics

eig_c, B_c = opg(Zc_tr, yc_tr, k=60, r=1, lam=1.0)
b1 = B_c[:, 0] * np.sign(np.corrcoef(Zc_tr @ B_c[:, 0], yc_tr)[0, 1])   # sign: mpg increases along b1

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].bar(np.arange(1, len(features) + 1), eig_c / eig_c.sum())
axes[0].set_title("Eigenvalues of M_hat")
axes[1].bar(features, b1); axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("First OPG direction")
axes[2].scatter(Zc_tr @ b1, yc_tr, s=10, alpha=0.6)
axes[2].set_xlabel("first OPG feature"); axes[2].set_ylabel("mpg")
axes[2].set_title("Response against the first OPG feature")
plt.tight_layout()
plt.show()

In [ ]:
ones = lambda A: np.column_stack([np.ones(len(A)), A])
theta_ols = ridge_fit(ones(Zc_tr), yc_tr)
k = 20
mse_ols = np.mean((ones(Zc_te) @ theta_ols - yc_te) ** 2)
mse_raw = np.mean((knn_regress(Zc_tr, yc_tr, Zc_te, k) - yc_te) ** 2)
mse_opg = np.mean((knn_regress(Zc_tr @ b1[:, None], yc_tr, Zc_te @ b1[:, None], k) - yc_te) ** 2)
print(f"Test MSE   OLS (6 features): {mse_ols:.2f}   kNN (6 features): {mse_raw:.2f}   "
      f"kNN (1 OPG feature): {mse_opg:.2f}")

beta_ols = theta_ols[1:] / np.linalg.norm(theta_ols[1:])
print(f"|cosine| between the first OPG direction and the OLS coefficients: {abs(b1 @ beta_ols):.2f}")

💬 **Question 3.4.**

1. How many directions stand out in the eigenvalues? Which features drive the first direction? Interpret the signs.
2. Look at the third panel. Why does OLS perform worse than kNN here?
3. Why does kNN on a single OPG feature beat kNN on the six features?
4. Use Paper question 3.2: for a linear model $f(x) = a + \beta^\top x$, what is $M$, and what does OPG return? The first OPG direction is only partly aligned with the OLS coefficients here. Why?

*Your answer:*

---
## Summary

* **SVM.** Maximise the margin. The soft margin is a hinge loss plus a penalty, like Ridge, with $C \approx 1/\lambda$. The solution depends on the support vectors only.
* **Kernel trick.** The SVM only needs inner products. Replacing them by a kernel $K(x, x') = \langle \varphi(x), \varphi(x') \rangle$ gives a linear SVM in a large feature space, computed in the original space. We checked that both routes give the same classifier, at a cost that does not depend on the size of the feature space. The Gaussian kernel even corresponds to an infinite feature space. $C$ and $\sigma$ control complexity and are chosen by validation.
* **kNN.** No training, only distances. $k$ controls the bias-variance trade-off. Standardise the features. Performance collapses when $d$ grows.
* **Local linear regression** is OLS on a neighbourhood. Its intercept estimates $f(x_0)$ and its slope estimates $\nabla f(x_0)$.
* **OPG.** If $f$ depends on $x$ only through $B^\top x$, all gradients lie in $\mathrm{span}(B)$. The leading eigenvectors of the averaged outer product of gradients recover it. Projecting before kNN removes the curse of dimensionality.

### To go further

* B. Schölkopf, A. Smola (2002). *Learning with Kernels.* MIT Press. The reference book on kernel methods.
* Y. Xia, H. Tong, W. K. Li, L.-X. Zhu (2002). *An adaptive estimation of dimension reduction space.* JRSS B. Introduces OPG and MAVE.
* K.-C. Li (1991). *Sliced inverse regression for dimension reduction.* JASA. Another classical method, based on inverse regression rather than gradients.
* P. Constantine (2015). *Active Subspaces.* SIAM. The same matrix $M$ is used to reduce the dimension of expensive numerical simulations.